In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

## Ingestion del archivo "movie_cast.json"

###Paso 1 - Leer el archivo JSON usando "DataframeReader" de Spark

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
movie_cast_schema = StructType([
    StructField("castOrder", IntegerType(), True),
    StructField("characterName", StringType(), True),
    StructField("genderId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("personId", IntegerType(), True)
    ])

In [0]:
movie_cast_df = spark.read\
    .option("multiline", True)\
    .schema(movie_cast_schema)\
    .json(f"{bronze_folder_path}/{v_file_date}/movie_cast.json")

### Paso 2 - Renombrar las columnas y añadir nuevas columnas

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
movie_cast_final_df = add_ingestion_date(movie_cast_df)\
    .drop("castOrder", "genderId")\
    .withColumnsRenamed({"movieId": "movie_id",
                         "personId": "person_id",
                         "characterName": "character_name"})\
    .withColumn("environment", lit("Production"))\
    .withColumn("file_date", lit(v_file_date))

### Paso 3 - Escribir la salida en un formato "Parquet" PartitionBy

In [0]:
#overwrite_partition("movie_silver", "movie_cast", "file_date", v_file_date)

In [0]:
#movie_cast_final_df.write.mode("overwrite").parquet(f"{silver_folder_path}/movie_cast")

In [0]:
#movie_cast_final_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.movie_cast")
condition_merge = 'tgt.movie_id = src.movie_id AND tgt.person_id = src.person_id AND tgt.file_date = src.file_date'

incremental_merge("movie_silver", "movie_cast", movie_cast_final_df, condition_merge, "file_date")

In [0]:
%sql
SELECT file_date, count(1)
FROM movie_silver.movie_cast
GROUP BY file_date;

file_date,count(1)
2024-12-16,30000
2024-12-23,15000
2024-12-30,5000


In [0]:
dbutils.notebook.exit("Exitoso")